# 04 Gold - Healthcare

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Builds the industry KPIs from accepted Silver records.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'healthcare':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

from pyspark.sql import functions as F

silver = {"appointments": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/appointments/", "encounters": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/encounters/", "patients": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/patients/", "providers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/providers/"}
gold = {"patient_utilization": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/healthcare/healthcare_patient_utilization/", "provider_daily": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/healthcare/healthcare_provider_daily/"}
patients = spark.table(table("silver", "patients"))
providers = spark.table(table("silver", "providers"))
appointments = spark.table(table("silver", "appointments"))
encounters = spark.table(table("silver", "encounters"))
appointment_metrics = (appointments.groupBy("participant_key", "patient_id")
    .agg(F.count("appointment_id").alias("appointment_count"), F.sum(F.when(F.col("status") == "no_show", 1).otherwise(0)).alias("no_show_count")))
encounter_metrics = (encounters.groupBy("participant_key", "patient_id")
    .agg(F.count("encounter_id").alias("encounter_count"), F.sum("cost_amount").alias("total_cost"), F.max("encounter_start").alias("last_encounter_at")))
patient_utilization = patients.select("participant_key", "patient_id").join(appointment_metrics, ["participant_key", "patient_id"], "left").join(encounter_metrics, ["participant_key", "patient_id"], "left")
patient_utilization.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("gold", "patient_utilization"))
scheduled = (appointments.withColumn("service_date", F.to_date("scheduled_start")).groupBy("participant_key", "service_date", "provider_id").agg(F.count("appointment_id").alias("scheduled_appointments"), F.sum(F.when(F.col("status") == "completed", 1).otherwise(0)).alias("completed_appointments"), F.avg(F.when(F.col("status") == "no_show", 1).otherwise(0)).alias("no_show_rate")))
performed = (encounters.withColumn("service_date", F.to_date("encounter_start")).withColumn("duration_minutes", (F.col("encounter_end").cast("long") - F.col("encounter_start").cast("long")) / 60).groupBy("participant_key", "service_date", "provider_id").agg(F.count("encounter_id").alias("encounter_count"), F.avg("duration_minutes").alias("average_duration_minutes"), F.sum("cost_amount").alias("total_cost")))
provider_daily = scheduled.join(performed, ["participant_key", "service_date", "provider_id"], "left")
provider_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("gold", "provider_daily"))
patient_utilization.show(20, truncate=False)

for table_name in gold:
    row_count = spark.table(table("gold", table_name)).count()
    assert row_count > 0, f"Gold table {table_name} is empty"
    print(f"Gold {table_name}: {row_count} rows")


## Expected result

Two non-empty, industry-specific aggregate Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
